# Situação 4: Limpeza do export do CRM (pandas)

O time de CRM exportou os clientes cadastrados desde julho para uma campanha, mas o arquivo veio com problemas: duplicatas, uma linha de teste, cidades, estados, datas, canais e e-mails sem padrão.

**Objetivo:** deixar o arquivo limpo e descobrir quantos clientes únicos existem e quantos estão sem e-mail.

In [1]:
import pandas as pd
import unicodedata
df = pd.read_csv('/content/clientes_crm_export_bruto.csv')

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2535 entries, 0 to 2534
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   cliente_id       2535 non-null   int64
 1   nome             2535 non-null   str  
 2   email            2440 non-null   str  
 3   cidade           2535 non-null   str  
 4   estado           2535 non-null   str  
 5   data_cadastro    2535 non-null   str  
 6   canal_aquisicao  2535 non-null   str  
dtypes: int64(1), str(6)
memory usage: 138.8 KB


In [3]:
df = df.drop_duplicates('cliente_id')
df = df[df['email'] != 'teste@teste.com']
df

,cliente_id,nome,email,cidade,estado,data_cadastro,canal_aquisicao
0,19915,Letícia Barbosa,leticia.barbosa19915@exemplo.com,Salvador,BA,27/10/2026,Orgânico
1,17888,Mariana Barbosa,mariana.barbosa17888@exemplo.com,Brasília,DF,2026-07-14,Orgânico
2,19985,Gabriela Gomes,NaN,Goiânia,GO,2026-10-30,Indicação
3,18066,Lucas Lima,lucas.lima18066@exemplo.com,Sao Paulo,SP,2026-07-24,Instagram
4,17636,Natália Almeida,natalia.almeida17636@exemplo.com,Brasília,DF,02/07/2026,Orgânico
...,...,...,...,...,...,...,...
2528,18529,Letícia Lima,leticia.lima18529@exemplo.com,rio de janeiro,RJ,2026-08-18,Instagram
2529,17821,Rafael Vieira,rafael.vieira17821@exemplo.com,São Paulo,SP,2026-07-11,Google Ads
2530,19213,Ana Rocha,ana.rocha19213@exemplo.com,Curitiba,PR,2026-09-20,Instagram
2531,18147,Mariana Lopes,mariana.lopes18147@exemplo.com,Belém,PA,2026-07-28,Orgânico


In [4]:
df['estado'] = df['estado'].str.upper()

In [5]:
def padronizar_cidade(cidade):
    if pd.isna(cidade):
        return cidade

    cidade = str(cidade).strip().upper()
    cidade = " ".join(cidade.split())

    cidade = unicodedata.normalize('NFKD', cidade)
    cidade = ''.join(
        letra for letra in cidade
        if not unicodedata.combining(letra)
    )

    return cidade

df['cidade'] = df['cidade'].apply(padronizar_cidade)

In [6]:
datas = df['data_cadastro'].astype('string').str.strip()

iso = pd.to_datetime(
    datas,
    format='%Y-%m-%d',
    errors='coerce'
)

br = pd.to_datetime(
    datas,
    format='%d/%m/%Y',
    errors='coerce'
)

df['data_cadastro'] = iso.fillna(br)

In [7]:
df['canal_aquisicao'] = df['canal_aquisicao'].replace({
    'google ads': 'Google Ads',
    'Organico': 'Orgânico',
    'insta': 'Instagram',
    'indicacao': 'Indicação',
    'email': 'E-mail'
})

In [8]:
df['email'] = df['email'].str.strip().str.lower()

In [9]:
email_nulos = df[df['email'].isna()]

quantidade = len(email_nulos)

print(f'Total de clientes com email nulo: {quantidade}')

email_nulos

Total de clientes com email nulo: 95


,cliente_id,nome,email,cidade,estado,data_cadastro,canal_aquisicao
2,19985,Gabriela Gomes,NaN,GOIANIA,GO,2026-10-30,Indicação
32,17670,Thiago Alves,NaN,RECIFE,PE,2026-07-04,Google Ads
44,17823,Bruno Nascimento,NaN,RIO DE JANEIRO,RJ,2026-07-11,Google Ads
50,19968,Henrique Oliveira,NaN,RECIFE,PE,2026-10-29,Google Ads
54,18294,Bianca Alves,NaN,RIO DE JANEIRO,RJ,2026-08-06,Indicação
...,...,...,...,...,...,...,...
2368,19866,Caio Martins,NaN,BELO HORIZONTE,MG,2026-10-24,Google Ads
2375,18524,Leonardo Nascimento,NaN,FORTALEZA,CE,2026-08-17,Instagram
2377,19841,Otávio Fernandes,NaN,BELEM,PA,2026-10-23,Instagram
2443,19542,Daniel Silva,NaN,RIO DE JANEIRO,RJ,2026-10-08,Google Ads


In [10]:
df.to_csv('clientes_crm_limpo.csv', index=False)

## Relatório para o time de CRM

- Arquivo original: **2.535 linhas**
- Removidas **143 duplicatas** (mesmo cliente exportado duas vezes) e **1 linha de teste**
- Padronizados: cidades, estados, datas (dois formatos misturados), canais (10 grafias → 5) e e-mails
- **Resultado: 2.391 clientes únicos, dos quais 95 estão sem e-mail** e não podem receber a campanha por e-mail.